# Retrieval-Ready Study Assistant — NCERT Science
## Week 9 Mini-Project · PG Diploma in AI-ML & Agentic AI Engineering

**Chapter used**: How Forces Affect Motion (NCERT Class 9 Science — Chapter 6)  
**Pipeline**: PDF Extraction → Cleaning → Chunking → BM25 Retrieval → Grounded Generation → Evaluation  
**Model**: phi3 via Ollama  

---
> **Before running**: make sure Ollama is running (`ollama serve`) and the model is pulled (`ollama pull phi3`).  
> Place the NCERT PDF at `Data/Raw/iesc106.pdf` (download from https://ncert.nic.in/textbook.php?iesc1=0-11).


---
## Stage 0 · Environment Setup

Install all required libraries. Run this once on a fresh clone.


In [ ]:
import subprocess, sys

packages = [
    "PyPDF2",
    "transformers",
    "torch",
    "rank_bm25",
    "ollama",
    "sentencepiece",   # required by T5 tokenizer
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")


In [ ]:
import os, re, sys, json, warnings
warnings.filterwarnings("ignore")

# Make sure src/ is on the path when notebook is at project root
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", PROJECT_ROOT)
print("src/ on path:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))


---
## Stage 1 · Corpus — Extract, Clean, Chunk, and Compare Tokenizers

### 1.1 PDF Extraction


In [ ]:
from src.extraction.pdf_loader import PDFLoader

PDF_PATH = os.path.join(PROJECT_ROOT, "Data", "Raw", "iesc106.pdf")

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH}\n"
        "Download Chapter 6 from https://ncert.nic.in/textbook.php?iesc1=0-11 "
        "and place it at Data/Raw/iesc106.pdf"
    )

loader = PDFLoader(PDF_PATH)
raw_text = loader.load()

print(f"Extracted {len(raw_text):,} characters from {len(raw_text.splitlines()):,} lines.")
print("\n--- First 500 characters of raw text ---")
print(raw_text[:500])


### 1.2 Text Cleaning

The raw PDF contains page-header artifacts (`Chapter-.indd`), figure captions, and special characters.  
We normalise whitespace, remove non-ASCII junk, and split on sentence boundaries.

> **Note**: The original `clean_text.py` had `re.sub(r'\b\d+\b', '', text)` which stripped ALL digits —  
> destroying physics numbers like "9.8 m/s²" and "F = ma". We fix that here by removing only  
> standalone page-number patterns instead.


In [ ]:
import re

def clean_text(text: str) -> str:
    # Remove PDF page-header artifacts like "Chapter-.indd -98Chapter-.indd -- ::-- ::"
    text = re.sub(r'Chapter-\.indd.*?::', '', text)

    # Collapse multiple whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove non-ASCII characters (keeps alphanumeric + common punctuation)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # Split on sentence boundaries for readability
    text = text.replace('. ', '.\n')

    return text.strip()

cleaned_text = clean_text(raw_text)

print(f"Cleaned text: {len(cleaned_text):,} characters")
print("\n--- First 500 characters of cleaned text ---")
print(cleaned_text[:500])


### 1.3 Content-Type Splitting

We split the corpus into three categories based on keyword patterns:
- **Concept paragraphs** — main explanatory body text
- **Worked examples** — sections starting with "Example" or "Activity"
- **End-of-chapter questions** — sections with "Exercise" or "Q." patterns


In [ ]:
def split_content_types(text: str) -> dict:
    lines = text.split('\n')
    concepts, examples, questions = [], [], []

    current_type = "concept"
    buffer = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        lower = line.lower()

        if re.match(r'^(example|activity|experiment)\b', lower):
            if buffer:
                concepts.append(' '.join(buffer))
                buffer = []
            current_type = "example"

        elif re.match(r'^(exercise|q\.|question|in-text question)', lower):
            if buffer:
                (examples if current_type == "example" else concepts).append(' '.join(buffer))
                buffer = []
            current_type = "question"

        buffer.append(line)

    # flush
    if buffer:
        target = {"concept": concepts, "example": examples, "question": questions}[current_type]
        target.append(' '.join(buffer))

    return {"concepts": concepts, "examples": examples, "questions": questions}

content = split_content_types(cleaned_text)

print(f"Concept paragraphs : {len(content['concepts'])}")
print(f"Worked examples    : {len(content['examples'])}")
print(f"Chapter questions  : {len(content['questions'])}")
print("\n--- Sample concept paragraph ---")
print(content['concepts'][0][:300] if content['concepts'] else "None found")


### 1.4 Tokenizer Comparison

We compare **GPT-2 BPE**, **BERT WordPiece**, and **T5 SentencePiece** on 5 representative passages  
from the chapter to understand how scientific terms are tokenized differently.


In [ ]:
from src.chunking.tokenizer import TokenizerComparison

tokenizer = TokenizerComparison()

# 5 representative passages from the chapter
passages = [
    "Force is a physical quantity that has both direction and magnitude, measured in newtons.",
    "Newton's second law states that F = ma, where F is the net force, m is the mass, and a is the acceleration.",
    "An object at rest remains at rest unless acted upon by a net external force — this is inertia.",
    "When two forces act on an object in opposite directions, the net force is the difference of their magnitudes.",
    "The acceleration produced by a force depends on the mass of the object and the magnitude of the force."
]

print(f"{'Passage':<70} {'GPT-2':>7} {'BERT':>7} {'T5':>7}")
print("-" * 95)

for p in passages:
    result = tokenizer.compare(p)
    short = p[:67] + "..." if len(p) > 67 else p
    print(f"{short:<70} {result['gpt2_tokens']:>7} {result['bert_tokens']:>7} {result['t5_tokens']:>7}")


In [ ]:
# Show token boundary differences on a scientific term
term = "specific-heat-capacity"
gpt2_tok = tokenizer.gpt2.tokenize(term)
bert_tok  = tokenizer.bert.tokenize(term)
t5_tok    = tokenizer.t5.tokenize(term)

print("Token boundary comparison for:", term)
print(f"  GPT-2 BPE       : {gpt2_tok}")
print(f"  BERT WordPiece  : {bert_tok}")
print(f"  T5 SentencePiece: {t5_tok}")

term2 = "photosynthesis"
print(f"\nToken boundary comparison for: {term2}")
print(f"  GPT-2 BPE       : {tokenizer.gpt2.tokenize(term2)}")
print(f"  BERT WordPiece  : {tokenizer.bert.tokenize(term2)}")
print(f"  T5 SentencePiece: {tokenizer.t5.tokenize(term2)}")


### 1.5 Chunking Strategy

**Parameters chosen: `chunk_size=300` words, `overlap=50` words, sentence-aligned boundaries.**

**Justification (150–250 words):**  
We use a sentence-aware chunking strategy rather than a purely fixed-size split. The sentence splitter 
(`re.split(r'(?<=[.!?])\s+', text)`) ensures chunks always end at natural sentence boundaries, 
preventing mid-sentence cuts that would confuse both BM25 scoring and the LLM.

The chunk size of 300 words was chosen after observing that a 200-word window split worked examples 
from their solutions — for instance, "Consider a car moving at 20 m/s" appeared in one chunk while 
the worked solution appeared in the next. BM25 then retrieved the problem but not the answer, causing 
the LLM to hallucinate a solution. At 300 words, most problem-solution pairs within the chapter stay 
together.

An overlap of 50 words carries the tail of the previous chunk into the next. This is important for 
transition sentences like "From the above experiment we can conclude..." which lose their meaning if 
they appear at the start of a chunk with no context. The overlap ensures BM25 can score these 
transition sentences correctly against queries that reference the conclusion.

We deliberately avoid 500-word chunks because the chapter sections themselves are only 200–400 words; 
a 500-word chunk would merge the First Law and Second Law explanations into one retrieval unit, 
diluting BM25 scores for law-specific queries.


In [ ]:
from src.chunking.chunker import Chunker

chunker = Chunker(cleaned_text, chunk_size=300, overlap=50)
chunks = chunker.create_chunks()

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c.split()) for c in chunks) // len(chunks)} words")
print(f"Min chunk length    : {min(len(c.split()) for c in chunks)} words")
print(f"Max chunk length    : {max(len(c.split()) for c in chunks)} words")

print("\n--- Chunk 0 ---")
print(chunks[0])
print("\n--- Chunk 1 ---")
print(chunks[1])


---
## Stage 2 · Retrieval — BM25 Chunk Store

We build a BM25Okapi retriever over the chunks.  
Each chunk carries metadata: chapter name, chunk index, and a word-count length.


In [ ]:
from src.retrieval.bm25 import BM25Retriever

retriever = BM25Retriever(chunks)

# Add metadata to each chunk for traceability
chunk_metadata = [
    {
        "chunk_id": i,
        "chapter": "Chapter 6 — How Forces Affect Motion",
        "word_count": len(c.split()),
        "text": c
    }
    for i, c in enumerate(chunks)
]

print(f"Chunk store built: {len(chunk_metadata)} chunks indexed")
print(f"Sample metadata entry:")
print(f"  chunk_id  : {chunk_metadata[0]['chunk_id']}")
print(f"  chapter   : {chunk_metadata[0]['chapter']}")
print(f"  word_count: {chunk_metadata[0]['word_count']}")
print(f"  text[:100]: {chunk_metadata[0]['text'][:100]}...")


### 2.1 Retrieval Test — 3 Queries

Test the retriever on three questions and confirm the returned chunks are relevant.


In [ ]:
test_queries = [
    "What is force?",
    "State Newton's Second Law of Motion.",
    "Why does a passenger fall forward when a bus stops suddenly?"
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    results = retriever.retrieve(q, top_k=3)
    for i, chunk in enumerate(results, 1):
        print(f"\n  [Chunk {i}] {chunk[:200]}...")


---
## Stage 3 · Generation — Grounded Answering with phi3

We use phi3 via Ollama for generation. The grounding prompt instructs the model to:  
1. Answer **only** from the provided context  
2. **Refuse** (say "I don't know") if the answer is not present in the context  

> Make sure Ollama is running: `ollama serve` in a separate terminal.


In [ ]:
from src.generation.llm import LLMGenerator

generator = LLMGenerator(model_name="phi3")

def answer(question: str):
    """
    Main callable: answer(question) -> (answer_str, retrieved_chunks)
    """
    retrieved = retriever.retrieve(question, top_k=3)
    context = "\n\n".join(retrieved)
    ans = generator.generate(question, context)
    return ans, retrieved


### 3.1 Grounding Prompt Versions

**v1 (original)** — permissive phrasing:
```
Answer ONLY from the provided context.
If answer is not in context, say "I don't know".
```

**v2 (recommended improvement)** — constraint phrasing:
```
If the answer is not present word-for-word in the context below,
respond with exactly: "I cannot answer this from the textbook."
```

The expert hint in the spec explicitly flags v1 as weaker — LLMs interpret "answer only from" as  
"prefer the context", whereas v2 forces an explicit check. Our evaluation results confirm this:  
17/20 answers were ungrounded despite the v1 instruction being present.


In [ ]:
# Test the answer() function on 3 sample questions
sample_questions = [
    "What is force?",
    "What is Newton's Third Law of Motion?",
    "What is photosynthesis?"   # out-of-scope — should refuse
]

for q in sample_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    ans, chunks = answer(q)
    print(f"A: {ans}")
    print(f"\n  Retrieved chunks (first 150 chars each):")
    for i, c in enumerate(chunks, 1):
        print(f"  [{i}] {c[:150]}...")


---
## Stage 4 · Evaluation

We evaluate 20 questions across three axes:
- **Correctness**: yes / partial / no
- **Grounding**: is the answer supported by retrieved chunks?
- **Refusal**: for out-of-scope questions, did the system refuse appropriately?

The evaluation set:
- 12 direct questions (from the textbook)
- 3 paraphrased questions
- 5 out-of-scope questions


In [ ]:
import json
from src.generation.llm import LLMGenerator

judge = LLMGenerator(model_name="phi3")

def evaluate_with_llm(question, ans, chunks):
    context = "\n\n".join(chunks)
    prompt = f"""
You MUST return JSON exactly like this:

{{
  "correctness": "yes/no/partial",
  "grounding": "yes/no",
  "refusal": "yes/no/na"
}}

Do not change keys. Do not add explanation.

Question: {question}
Answer: {ans}
Context: {context}
"""
    response = judge.generate(question="", context=prompt).strip()
    if "```" in response:
        response = response.split("```")[1]
    try:
        data = json.loads(response)
        return {
            "correctness": data.get("correctness", "partial"),
            "grounding":   data.get("grounding",   "no"),
            "refusal":     data.get("refusal",      "na")
        }
    except:
        return {"correctness": "partial", "grounding": "no", "refusal": "na"}


In [ ]:
questions = [
    # Direct (12)
    "What is force?",
    "What are the effects of force on an object?",
    "What is Newton's First Law of Motion?",
    "What is inertia?",
    "What are the different types of inertia?",
    "What is momentum?",
    "State Newton's Second Law of Motion.",
    "What is the formula for force according to Newton's Second Law?",
    "What is Newton's Third Law of Motion?",
    "What is action and reaction?",
    "Why does a passenger fall forward when a moving bus stops suddenly?",
    "Why do we pull our hands back quickly after touching a hot object?",

    # Paraphrased (3)
    "Define force in simple terms.",
    "Why does a body resist change in motion?",
    "Explain Newton's second law in your own words.",

    # Out-of-scope (5)
    "What is photosynthesis?",
    "Explain Ohm's Law.",
    "Who discovered gravity?",
    "What is the capital of India?",
    "Explain quantum entanglement in Chapter 6.",
]

question_types = (
    ["Direct"] * 12 +
    ["Paraphrased"] * 3 +
    ["Out-of-scope"] * 5
)


In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────
# NOTE: This cell calls Ollama 20 times (answers) + 20 times (judge) = 40 API calls.
# Expected runtime: 5–15 minutes depending on hardware.

eval_results = []

for q, qtype in zip(questions, question_types):
    ans, chunks = answer(q)
    scores = evaluate_with_llm(q, ans, chunks)

    print(f"\n======================")
    print(f"Q: {q}")
    print(f"A: {ans[:300]}{'...' if len(ans) > 300 else ''}")
    print(f"Eval: {scores}")

    eval_results.append({
        "question":    q,
        "type":        qtype,
        "answer":      ans,
        "correctness": scores["correctness"],
        "grounding":   scores["grounding"],
        "refusal":     scores["refusal"],
    })

print("\n✅ Evaluation complete.")


### 4.1 Summary Scores


In [ ]:
correct_yes    = sum(1 for r in eval_results if r["correctness"] == "yes")
correct_partial = sum(1 for r in eval_results if r["correctness"] == "partial")
correct_no     = sum(1 for r in eval_results if r["correctness"] == "no")
grounded       = sum(1 for r in eval_results if r["grounding"]   == "yes")
oos            = [r for r in eval_results if r["type"] == "Out-of-scope"]
refused        = sum(1 for r in oos if r["refusal"] == "yes")

total = len(eval_results)
print(f"Total questions  : {total}")
print(f"Correctness yes  : {correct_yes} / {total}  ({100*correct_yes//total}%)")
print(f"Correctness part : {correct_partial} / {total}  ({100*correct_partial//total}%)")
print(f"Correctness no   : {correct_no} / {total}  ({100*correct_no//total}%)")
print(f"Grounded         : {grounded} / {total}  ({100*grounded//total}%)")
print(f"Appropriate refs : {refused} / {len(oos)}  (out-of-scope only)")


### 4.2 Results Table


In [ ]:
# Pretty-print results table
header = f"{'#':<4} {'Question':<55} {'Type':<14} {'Correct':<10} {'Grounded':<10} {'Refusal'}"
print(header)
print("-" * len(header))
for i, r in enumerate(eval_results, 1):
    q_short = r["question"][:52] + "..." if len(r["question"]) > 52 else r["question"]
    print(f"{i:<4} {q_short:<55} {r['type']:<14} {r['correctness']:<10} {r['grounding']:<10} {r['refusal']}")


### 4.3 Failure Analysis

**3 Working Examples + 2 Failing Examples with probable cause.**


In [ ]:
working   = [(1, "What is force?",
              "Correct definition with magnitude/direction; grounded in retrieved chunk."),
             (8, "What is the formula for force (Newton 2nd Law)?",
              "F=ma stated correctly; retriever surfaced relevant chunk (grounding=yes) despite PDF noise appended."),
             (14,"Why does a body resist change in motion?",
              "Correctly cited inertia + Newton's First Law; factually accurate even though not grounded.")]

failing   = [(2, "What are the effects of force on an object?",
              "BM25 retrieved 'Curiosity Chapter' sidebar chunks; LLM reproduced sidebar Q&A format verbatim. "
              "Root cause: PDF extraction noise — fix is corpus cleaning, not prompt."),
             (11,"Why does a passenger fall forward when a moving bus stops suddenly?",
              "Retriever returned airbag/road-safety chunk (token overlap on 'moving', 'stop'); "
              "LLM generated from wrong chunk and hallucinated a handcart collision mid-answer.")]

print("WORKING EXAMPLES")
print("="*70)
for num, q, note in working:
    print(f"Q{num}: {q}")
    print(f"     {note}\n")

print("FAILING EXAMPLES")
print("="*70)
for num, q, note in failing:
    print(f"Q{num}: {q}")
    print(f"     {note}\n")


---
## End-to-End Demo

Quick interactive demo — run this cell to ask a single question.


In [ ]:
# Change this question to test your own queries
demo_question = "What is Newton's Second Law of Motion?"

ans, retrieved_chunks = answer(demo_question)

print(f"Question: {demo_question}")
print(f"\nAnswer:\n{ans}")
print(f"\n--- Retrieved Chunks ({len(retrieved_chunks)}) ---")
for i, c in enumerate(retrieved_chunks, 1):
    print(f"\n[{i}] {c[:300]}{'...' if len(c) > 300 else ''}")


---
## Summary

| Stage | Component | Status |
|-------|-----------|--------|
| 1 — Corpus | PDF extraction (PyPDF2) | ✅ |
| 1 — Corpus | Text cleaning (fixed digit bug) | ✅ |
| 1 — Corpus | Content-type split (concepts / examples / questions) | ✅ |
| 1 — Corpus | Tokenizer comparison (GPT-2 / BERT / T5) | ✅ |
| 1 — Corpus | Chunking (300 words, 50 overlap, sentence-aligned) | ✅ |
| 2 — Retrieval | BM25Okapi with chunk metadata | ✅ |
| 2 — Retrieval | 3-query retrieval test | ✅ |
| 3 — Generation | phi3 via Ollama, grounding prompt | ✅ |
| 3 — Generation | `answer(question) → (answer, chunks)` | ✅ |
| 4 — Evaluation | 20-question eval (12 direct / 3 paraphrased / 5 OOS) | ✅ |
| 4 — Evaluation | Three-axis scoring (correctness / grounding / refusal) | ✅ |
| 4 — Evaluation | Failure analysis (3 working + 2 failing) | ✅ |

See `evaluation_results.md` for the full scored table.  
See `reflection.md` for architecture discussion and honest self-assessment.
